# 05 · Generate answers (the full RAG pipeline)
Retrieves the most relevant chunks from ChromaDB and asks a local LLM (`llama3.2:3b` via Ollama)
to answer using only those chunks, citing its sources.

In [1]:
# ===== CELL 1: Setup =====
# Loads the saved vector database (no re-embedding needed), the embedding model,
# and reads whether this computer should run the LLM on the CPU only.

import os                                               # os: lets Python read settings from the computer (environment variables)
import time                                             # time: to measure how long each answer takes
import chromadb                                         # chromadb: our vector database
import ollama                                           # ollama: talks to the LLM running on this computer
from sentence_transformers import SentenceTransformer   # SentenceTransformer: embeds the questions

EMBED_MODEL = "BAAI/bge-small-en-v1.5"                  # Must be the SAME model used to embed the chunks, or the search breaks
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # The instruction this model expects in front of questions
LLM = "llama3.2:3b"                                     # The Ollama model that writes the answers

# Some computers (often with built-in graphics chips) produce garbage output when Ollama uses the GPU.
# On those computers, set the environment variable RAG_FORCE_CPU=1 to run the LLM on the CPU only.
# Everywhere else, leave it unset and Ollama picks the best hardware automatically.
FORCE_CPU = os.getenv("RAG_FORCE_CPU", "0") == "1"      # Read the setting; if it doesn't exist, use "0" (off). True only if it is "1"

embedder = SentenceTransformer(EMBED_MODEL)             # Load the embedding model (already downloaded, so this is quick)
client = chromadb.PersistentClient(path="../data/chroma")       # Open the database saved by 04_embed
collection = client.get_collection("aspnetcore_docs_bge_small") # Open our existing collection (get, not create)

print("Chunks in database:", collection.count())        # Should print 6601
print("LLM:", LLM)                                      # Which model will write the answers
print("Force CPU:", FORCE_CPU)                          # Which mode is active (True on computers where RAG_FORCE_CPU=1)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Chunks in database: 6601
LLM: llama3.2:3b
Force CPU: True


In [2]:
# ===== CELL 2: Retrieval =====
# Finds the k chunks closest in meaning to the question.

def retrieve(question, k=5):                            # question: what the user asked; k: how many chunks to fetch
    q = embedder.encode([QUERY_PREFIX + question], normalize_embeddings=True)   # Embed the question
    res = collection.query(query_embeddings=q.tolist(), n_results=k)            # Find the k closest chunks
    hits = []                                           # The results go here
    for cid, text, meta, dist in zip(res["ids"][0], res["documents"][0],
                                     res["metadatas"][0], res["distances"][0]):   # Walk through the results together
        hits.append({                                   # Save one result
            "chunk_id": cid,                            # Which chunk
            "score": round(1 - dist / 2, 3),            # Similarity (1.0 = identical meaning)
            "where": meta["title"] + (f" > {meta['section']}" if meta["section"] else ""),   # "Article > Section"
            "url": meta["url"],                         # Link to the page, for citing
            "text": text,                               # The passage itself
        })
    return hits                                         # Return the list

In [3]:
# ===== CELL 3: The prompt =====
# The instructions that keep the LLM grounded in the docs, and the function
# that packs the question and the retrieved chunks into one message.

SYSTEM_PROMPT = """You answer questions about ASP.NET Core 10 using ONLY the documentation excerpts provided.
Rules:
- Use only information from the excerpts. Do not use outside knowledge.
- If the excerpts do not contain the answer, reply exactly: "I couldn't find this in the documentation."
- Cite the excerpts you used by number in square brackets, like [1] or [2][3].
- Be concise: a short paragraph, plus a short code snippet only if an excerpt contains one."""
# The system prompt sets the rules the LLM must follow for every question

def build_prompt(question, hits):                       # Combine the retrieved chunks and the question into one message
    context = "\n\n".join(                              # Join all the chunks, separated by blank lines...
        f"[{i}] {h['where']}\n{h['text']}"              # ...each one numbered [1], [2]... with its "Article > Section" heading
        for i, h in enumerate(hits, start=1)            # enumerate counts the chunks, starting from 1
    )
    return f"Documentation excerpts:\n\n{context}\n\nQuestion: {question}\n\nAnswer:"   # The full message sent to the LLM

In [4]:
# ===== CELL 4: Ask a question =====
# retrieve -> build the prompt -> LLM writes the answer -> show answer + sources.

LLM_OPTIONS = {                                         # Generation settings, shared by every question
    "temperature": 0.1,                                 # Very low randomness: answers stay focused, but the model can escape a repeating loop
    "seed": 42,                                         # Fixed random seed, so the same question gives the same answer every run
    "repeat_penalty": 1.15,                             # Discourage reusing words it has just written (prevents loops)
    "num_predict": 400,                                 # Hard limit on answer length (~300 words), so it can never run forever
    "num_ctx": 4096,                                    # How much text the model may read at once (so no excerpts get cut off)
}
if FORCE_CPU:                                           # Only on computers where RAG_FORCE_CPU=1 is set...
    LLM_OPTIONS["num_gpu"] = 0                          # ...send zero model layers to the GPU, so the CPU does all the work

def ask(question, k=5):                                 # The complete RAG pipeline in one function
    hits = retrieve(question, k)                        # 1. Find the most relevant chunks
    prompt = build_prompt(question, hits)               # 2. Pack them into a prompt with the question
    start = time.time()                                 # Start the timer
    try:                                                # Try to get an answer; if the LLM fails, handle it below
        response = ollama.chat(                         # 3. Send it to the LLM
            model=LLM,                                  # Which model to use
            messages=[                                  # The conversation: rules first, then the question with excerpts
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            options=LLM_OPTIONS,                        # The settings above
        )
        answer = response["message"]["content"]         # The LLM's reply text
    except ollama.ResponseError as e:                   # If Ollama reports an error...
        answer = f"[Generation failed: {e}]"            # ...record it as the answer instead of crashing
    return {                                            # 4. Return everything we need for display and, later, evaluation
        "question": question,
        "answer": answer,
        "hits": hits,                                   # The chunks it was given
        "seconds": round(time.time() - start, 1),       # How long the LLM took
    }

def show_answer(result):                                # Print an answer with its numbered sources
    print("Q:", result["question"], "\n")               # The question
    print(result["answer"], "\n")                       # The answer
    print("Sources:")                                   # Then the sources it was given
    for i, h in enumerate(result["hits"], start=1):     # Number them the same way as in the prompt
        print(f"  [{i}] {h['where']}  ({h['score']})")  # Where each chunk came from, and its search score
    print(f"\n({result['seconds']} s)")                 # How long it took
    print("=" * 100)                                    # A separator line

In [5]:
# ===== CELL 5: Try it =====
# Two questions retrieval got right, the one it missed, and one that isn't in the docs at all.

show_answer(ask("What does the @key directive attribute do in Blazor?"))
show_answer(ask("How do I apply a filter to all controllers globally?"))
show_answer(ask("How do I register a service so a new instance is created per HTTP request?"))
show_answer(ask("How do I deploy a Django app to Heroku?"))

Q: What does the @key directive attribute do in Blazor? 

The `@key` directive attribute is used to control the mapping of elements or components to a collection. It guarantees the preservation of elements or components based on the key's value, and can be used to prevent unnecessary rerendering when the underlying data changes.

For example:

```razor
<Details @key="person" Data="@person.Data" />
```

This ensures that only `Details` components with a matching `person` object are re-rendered. 

Sources:
  [1] Retain element, component, and model relationships in ASP.NET Core Blazor > Use of the `@key` directive attribute  (0.878)
  [2] Retain element, component, and model relationships in ASP.NET Core Blazor > Use of the `@key` directive attribute  (0.873)
  [3] Retain element, component, and model relationships in ASP.NET Core Blazor > Use of the `@key` directive attribute  (0.859)
  [4] Retain element, component, and model relationships in ASP.NET Core Blazor > Scope of `@key  (0.85

In [6]:
# ===== TEST: run the model on the CPU only =====
# "num_gpu": 0 tells Ollama not to use the graphics chip at all.
r = ollama.chat(
    model=LLM,                                           # Same model as before
    messages=[{"role": "user", "content": "Explain dependency injection in one sentence."}],   # A simple question, no documents
    options={"num_gpu": 0, "temperature": 0.1, "num_predict": 60},   # CPU only, low randomness, short answer
)
print(r["message"]["content"])                           # Print the reply

Dependency injection is a design pattern that allows components to be loosely coupled, making it easier to test, maintain, and extend software systems by injecting dependencies rather than hardcoding them.
